# 03 - Applying sampled parameters to a model config

This notebook confirms how a tuner turns sampled parameters into concrete
overrides on a fresh model configuration object. It exercises the
`_apply_params` logic of `Phase1Tuner` and `Phase2Tuner` (from
`pipelines.tuning_pipeline.tuners`) and shows the before/after diff of the
configuration fields that change.

The tuners require trainer/dataset configs and a logger to instantiate fully,
but `_apply_params` only needs a sampler and the model config class. We build a
minimal stand-in so the override mechanism can be exercised on CPU without any
training.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import matplotlib.pyplot as plt
import optuna

import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.rcParams.update({
    "figure.figsize" : (7.0, 4.2),
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
})


## A minimal harness around _apply_params

We instantiate the real `Phase1Tuner` / `Phase2Tuner` with placeholder configs
and a no-op logger. Only `_apply_params`, `model_config_cls` and `sampler` are
touched, all of which are set in `BaseTuner.__init__`.

In [ ]:
from pipelines.tuning_pipeline.tuners import Phase1Tuner, Phase2Tuner
from configuration.models_config    import UNetConfig
from configuration.tuning_config     import Phase1TuneConfig, Phase2TuneConfig

class NullLogger:
    def __getattr__(self, _):
        return lambda *a, **k: None

common = dict(
    model_name          = 'unet',
    model_config_cls    = UNetConfig,
    base_trainer_config = None,
    base_dataset_config = None,
    log_dir             = '/tmp/nb_tuning_unused',
    logger              = NullLogger(),
)

p1 = Phase1Tuner(tune_cfg=Phase1TuneConfig(), **common)
p2 = Phase2Tuner(tune_cfg=Phase2TuneConfig(), best_phase1_params={}, **common)
print('tuners constructed:', type(p1).__name__, type(p2).__name__)

## Phase-1 override diff

On a fresh `UNetConfig`, Phase 1 overrides the optimisation fields. We snapshot
the defaults, apply the sampled parameters from one trial, and report the
fields that changed.

In [ ]:
import dataclasses

def field_snapshot(cfg, names):
    return {n: getattr(cfg, n) for n in names}

def diff(before, after):
    return {k: (before[k], after[k]) for k in before if before[k] != after[k]}

lr_names = list(UNetConfig.tunable_lr_params().keys())

def p1_objective(trial):
    cfg    = UNetConfig()
    before = field_snapshot(cfg, lr_names)
    p1._apply_params(trial, cfg)
    after  = field_snapshot(cfg, lr_names)
    trial.set_user_attr('changed', list(diff(before, after).keys()))
    p1_objective.last = (before, after)
    return 0.0

study1 = optuna.create_study(sampler=optuna.samplers.RandomSampler(seed=SEED))
study1.optimize(p1_objective, n_trials=1)

before, after = p1_objective.last
for k, (b, a) in diff(before, after).items():
    print(f'  {k:18s} {b:.3e} -> {a:.3e}' if isinstance(a, float) else f'  {k:18s} {b} -> {a}')
print('changed fields:', len(diff(before, after)), 'of', len(lr_names))

## Phase-2 override diff with a fixed Phase-1 best

Phase 2 first writes back the best Phase-1 parameters (here a small fixed dict)
and then overrides the architecture fields with its own sampled values. The
plot shows which configuration fields the combined application touches.

In [ ]:
p2.best_phase1_params = {'encoder_lr': 1.234e-3, 'dropout': 0.42}
arch_names = ['features', 'bottleneck_factor', 'activation', 'normalization', 'upsample_mode']
track      = lr_names + arch_names

def p2_objective(trial):
    cfg    = UNetConfig()
    before = field_snapshot(cfg, track)
    p2._apply_params(trial, cfg)
    after  = field_snapshot(cfg, track)
    p2_objective.last = (before, after)
    return 0.0

study2 = optuna.create_study(sampler=optuna.samplers.RandomSampler(seed=SEED))
study2.optimize(p2_objective, n_trials=1)

before, after = p2_objective.last
changed = diff(before, after)
for k, (b, a) in changed.items():
    print(f'  {k:18s} {b} -> {a}')

fig, ax = plt.subplots(figsize=(7, 4.5))
flags = [1 if n in changed else 0 for n in track]
colors = ['darkorange' if n in arch_names else 'steelblue' for n in track]
ax.barh(track, flags, color=colors)
ax.set_xlim(0, 1.2)
ax.set_xticks([0, 1])
ax.set_xticklabels(['unchanged', 'changed'])
ax.set_title('Phase-2 fields touched (blue=from P1 best, orange=arch sampled)')
ax.invert_yaxis()
fig.tight_layout()
plt.show()

## Expected visual outcome

The Phase-1 diff lists nine optimisation fields, each moving from its default to
a sampled value. The Phase-2 chart shows the two injected Phase-1 best fields
(`encoder_lr`, `dropout`) plus the five architecture fields marked as changed,
while the remaining Phase-1 fields stay at their defaults.